# Weekly Project 5 
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.


### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

# TASK 1

In [ ]:
import open3d as o3d
import numpy as np
import copy

# helper function for drawing
def draw_registrations(source, target, transformation = None, recolor = False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if(recolor):
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if(transformation is not None):
        source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

### Match R1 and R2

The first step is to get the point cloud, the o3d.io.read_point_cloud function loads both the source and target point cloud. Also is importanto to include a variable (voxel_size_R) to downscale these point clouds to reduce computational complexity and noise.

In [ ]:
source_R = o3d.io.read_point_cloud("global_registration/r1.pcd")
target_R = o3d.io.read_point_cloud("global_registration/r2.pcd")
voxel_size_R = 0.05


The next step is to preprocess the point cloud. This function first applies the voxel downsampling defined previously; then it computes local surface normals to get the essential geometric baseline needed to compute FPFH (Fast Point Feature Histograms) descriptors; and then it calculates the FPFH, which encode the surrounding surface geometry into distinct vectors used by RANSAC to identify correspondences and align the point clouds.

In [ ]:
#Function to preprocess the point cloud (downsampling, normals, and FPFH)
def preprocess_point_cloud(pcd, voxel_size):
    pcd_down = pcd.voxel_down_sample(voxel_size)
    
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30)
    )
    # radius: Maximum spherical search radius.
    # #max_nn: Maximum number of nearest neighbors to consider within that radius.
    
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100)
    )
    
    return pcd_down, pcd_fpfh

# Apply preprocessing to both source and target
source_down_R, source_fpfh_R = preprocess_point_cloud(source_R, voxel_size_R)
target_down_R, target_fpfh_R = preprocess_point_cloud(target_R, voxel_size_R)

This step performs global registration using the RANSAC algorithm to find an initial alignment, not a perfect one, using a globan point to point method, which is faster and give better resoults than the point to plane. It is drawn the original point clouds, not the downsampled ones, to see the final result of the registration.

In [ ]:
distance_threshold = voxel_size_R * 1.5 #maximum allowed Euclidean distance to consider two points matched

point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False)

ransac_result_R = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down_R, target_down_R,
    source_fpfh_R, target_fpfh_R,
    True,
    distance_threshold,
    point_to_point,
    )

draw_registrations(source_R, target_R, ransac_result_R.transformation, True)

### Car 1 and Car 2

The same is done for the images Car1 and Car2, only changing some values to get a better result, such as the voxel size.

In [ ]:
source_car = o3d.io.read_point_cloud("global_registration/car1.ply")
target_car = o3d.io.read_point_cloud("global_registration/car2.ply")

voxel_size_car = 0.1

In [ ]:
def preprocess_point_cloud(pcd, voxel_size):
    pcd_down = pcd.voxel_down_sample(voxel_size)
    
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30)
    )
    
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100)
    )
    
    return pcd_down, pcd_fpfh

source_down_car, source_fpfh_car = preprocess_point_cloud(source_car, voxel_size_car)
target_down_car, target_fpfh_car = preprocess_point_cloud(target_car, voxel_size_car)

In [ ]:
distance_threshold_car = voxel_size_car * 1.5 

point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane =  o3d.pipelines.registration.TransformationEstimationPointToPlane()

ransac_result_car = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down_car, target_down_car,
    source_fpfh_car, target_fpfh_car,
    True,
    distance_threshold_car,
    point_to_point,
    )

draw_registrations(source_car, target_car, ransac_result_car.transformation, True) 

# Task 2

RANSAC is great at global registration, but the alignment is still pretty rough. After applying RANSAC is useful to use local registration, that is why ICP (Iterative Closest Point). The tool is perfect for getting the cloud points into place with high precision.

To get a good result we set a sets distance threshold so only very close points are matched, and it uses "ransac_result_R.transformation" as the initial transformation estimate instead of an identity matrix. Now, by applying the point-to-plane approach, ICP minimizes the orthogonal distance to the surface normals, quickly fine-tuning the alignment.
This is done for both images from before.

In [ ]:
icp_distance_threshold = voxel_size_R * 0.4

icp_result_R = o3d.pipelines.registration.registration_icp(
    source_down_R, target_down_R, 
    icp_distance_threshold,
    ransac_result_R.transformation, #instead of np.identity(4), we use the transformation from RANSAC as the initial guess for ICP
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

print("Transformación Refinada (ICP Local):")
print(icp_result_R.transformation)

draw_registrations(source_R, target_R, icp_result_R.transformation, True)

In [ ]:
icp_distance_threshold = voxel_size_car * 0.4

icp_result_car = o3d.pipelines.registration.registration_icp(
    source_down_car, target_down_car, 
    icp_distance_threshold,
    ransac_result_car.transformation,
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

print("Transformación Refinada (ICP Local):")
print(icp_result_car.transformation)

draw_registrations(source_car, target_car, icp_result_car.transformation, True)

## Reconstruction

Instead of using RANSAC for every single pair, we do sequential frame-to-frame reconstruction.The get_pcd_from_rgbd function lifts paired color and depth maps into metric 3D point clouds using pinhole camera intrinsics. Because consecutive camera shots barely move relative to each other, identity is already a solid starting estimate for ICP. Each relative pairwise pose is multiplied into a global transformation chain (current_transformation @ reg_p2l.transformation) to register the current frame into the coordinate origin of frame zero. We continuously accumulate the camera's trajectory, paste each aligned frame into one master point cloud, and downsample as we go so the computer's memory doesn't get overloaded.

The final result is not optimal because we were unable to properly optimize the newly obtained transformation matrices throughout the sequence. Simply chaining the local transformations without applying a global refinement or pose graph optimization caused small estimation errors in each matrix to accumulate and drift, resulting in noticeable misalignment across the model.

In [65]:
# Helper function to convert RGB and Depth image pairs into a PointCloud
def get_pcd_from_rgbd(color_path, depth_path):

    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_path,
        depth_path,
        depth_trunc=2, #to get rid of the background noise
        convert_rgb_to_intensity=False,
    )

    camera = o3d.camera.PinholeCameraIntrinsic(
        o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault)

    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(
        rgbd, camera)

    # Correct default sensor orientation
    pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
    return pcd


# Paths and image sequence configuration
base_path = "car_challange"
num_frames = 1700 # Target frame limit
step = 2 # Frame skip step to balance overlap and runtime
voxel_size = 0.02
threshold = 0.05
trans_init = np.identity(4)

frame_0_color = o3d.io.read_image(f"{base_path}/rgb/0000001.jpg")
frame_0_depth = o3d.io.read_image(f"{base_path}/depth/0000001.png")

target_pcd = get_pcd_from_rgbd(frame_0_color, frame_0_depth)
target_pcd = target_pcd.voxel_down_sample(voxel_size=0.05)

reconstruction = copy.deepcopy(target_pcd)
current_transformation = np.identity(4)

# Estimate normals on initial target point cloud
target_pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30)
)

current_transformation = np.identity(4)

# Iterate through the image sequence
for i in range(step, num_frames, step):
    filename_color = o3d.io.read_image(f"{base_path}/rgb/{i:07d}.jpg")
    filename_depth = o3d.io.read_image(f"{base_path}/depth/{i:07d}.png")

    source_pcd = get_pcd_from_rgbd(filename_color, filename_depth)
    source_pcd = source_pcd.voxel_down_sample(voxel_size=voxel_size)

    # Estimate normals required for point-to-plane ICP
    source_pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(
            radius=0.05, max_nn=30
        )
    )
    target_pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(
            radius=0.05, max_nn=30
        )
    )

    reg_p2l = o3d.pipelines.registration.registration_icp(
        source_pcd,
        target_pcd,
        threshold,
        trans_init,
        o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    )

    # Accumulate the global transformation matrix
    current_transformation = current_transformation @ reg_p2l.transformation


    # Fuse it into the global reconstruction 
    source_transformed = copy.deepcopy(source_pcd)
    source_transformed.transform(current_transformation)
    reconstruction += source_transformed
    reconstruction = reconstruction.voxel_down_sample(voxel_size=0.02)


    # Update target for the next iteration
    target_pcd = copy.deepcopy(source_pcd)

    print(f"Frame {i:07d} integrated. Fitness: {reg_p2l.fitness:.2f} | Total points: {len(reconstruction.points)}")

o3d.visualization.draw_geometries([reconstruction])

Frame 0000002 integrated. Fitness: 1.00 | Total points: 3508
Frame 0000004 integrated. Fitness: 0.98 | Total points: 4242
Frame 0000006 integrated. Fitness: 1.00 | Total points: 4424
Frame 0000008 integrated. Fitness: 1.00 | Total points: 4759
Frame 0000010 integrated. Fitness: 1.00 | Total points: 4940
Frame 0000012 integrated. Fitness: 1.00 | Total points: 4384
Frame 0000014 integrated. Fitness: 1.00 | Total points: 4667
Frame 0000016 integrated. Fitness: 1.00 | Total points: 4996
Frame 0000018 integrated. Fitness: 1.00 | Total points: 5273
Frame 0000020 integrated. Fitness: 1.00 | Total points: 4688
Frame 0000022 integrated. Fitness: 1.00 | Total points: 4897
Frame 0000024 integrated. Fitness: 1.00 | Total points: 4838
Frame 0000026 integrated. Fitness: 1.00 | Total points: 5053
Frame 0000028 integrated. Fitness: 1.00 | Total points: 4942
Frame 0000030 integrated. Fitness: 1.00 | Total points: 5048
Frame 0000032 integrated. Fitness: 1.00 | Total points: 5241
Frame 0000034 integrated

AI DECLARATION:

We used Gemini and Claude to help regarding definitions of functions and arguments and general debugging.